# Challenge A: PyTorch Implementation (Antonio Sainz)

**Gradient Descent Workshop · CSCN8010 · Group 1 · Antonio Sainz · 9072844**

This notebook covers the PyTorch part of the workshop:

1. Challenge A — manual gradient descent with autograd, with each of the five training steps explained.
2. What goes wrong when `w.grad.zero_()` is removed.
3. Challenge A rewritten with `torch.optim.SGD` and `nn.MSELoss`, checked against the manual version.
4. Challenge A trained with three learning rates.
5. Reflection questions Q6, Q9 and Q10, plus insights from these runs.

**Data.** The four points used by the workshop, $x = [1, 2, 3, 4]$ and $y = [2, 4, 6, 8]$. They follow $y = 2x$ exactly, so the ideal weight is $w = 2$ and the ideal loss is 0. The model has a single weight and no bias: $\hat{y} = xw$.

In [1]:
import torch
import torch.nn as nn
import plotly.graph_objects as go

print("PyTorch version:", torch.__version__)

# Workshop data for Challenge A: y = 2x
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[2.0], [4.0], [6.0], [8.0]])

# Same starting weight for every experiment, so the comparisons are fair
torch.manual_seed(0)
w_init = torch.randn(1, 1)
print("Initial weight:", round(w_init.item(), 4))

PyTorch version: 2.11.0+cpu
Initial weight: 1.541


## 1. Challenge A — manual gradient descent with autograd

For the loss $L(w) = \frac{1}{n}\sum_i (x_i w - y_i)^2$ the gradient is $\frac{\partial L}{\partial w} = \frac{2}{n}\sum_i x_i (x_i w - y_i)$. PyTorch computes this value for us through autograd. Each epoch runs the same five steps:

| Step | Code | What it does |
|---|---|---|
| 1. Forward pass | `y_hat = x @ w` | Predicts with the current weight; PyTorch records the operation in a graph because `w` has `requires_grad=True`. |
| 2. Loss | `((y_hat - y) ** 2).mean()` | Measures the error as mean squared error (MSE), a single number. |
| 3. Backward | `loss.backward()` | Walks the graph backwards and stores $\partial L / \partial w$ in `w.grad`. |
| 4. Update | `w -= lr * w.grad` inside `torch.no_grad()` | Moves the weight against the gradient. `no_grad` stops the update itself from being recorded as part of the model. |
| 5. Reset | `w.grad.zero_()` | Clears the stored gradient, because PyTorch adds new gradients to the old ones. |

The training loop is written as a function so the same code can be reused for the next experiments.

In [2]:
def train_manual(lr=0.01, epochs=200, reset_grad=True):
    # Returns the learned weight and the loss of every epoch
    w = w_init.clone().requires_grad_(True)
    losses = []
    for epoch in range(epochs):
        y_hat = x @ w                        # 1. forward pass: prediction with the current weight
        loss = ((y_hat - y) ** 2).mean()     # 2. MSE loss
        loss.backward()                      # 3. autograd fills w.grad with dL/dw

        with torch.no_grad():                # 4. update outside the graph
            w -= lr * w.grad

        if reset_grad:
            w.grad.zero_()                   # 5. clear the gradient for the next epoch
        losses.append(loss.item())
    return w.detach(), losses


w_manual, loss_manual = train_manual(lr=0.01, epochs=200)
print(f"Learned weight: {w_manual.item():.6f}   (true value 2)")
print(f"Loss at epoch 1:   {loss_manual[0]:.6f}")
print(f"Loss at epoch 200: {loss_manual[-1]:.2e}")

Learned weight: 2.000000   (true value 2)
Loss at epoch 1:   1.580135
Loss at epoch 200: 8.99e-13


In [3]:
fig = go.Figure(go.Scatter(x=list(range(1, 201)), y=loss_manual, mode="lines", name="lr = 0.01"))
fig.update_layout(title="Challenge A: training loss (manual gradient descent)",
                  xaxis_title="Epoch", yaxis_title="MSE loss (log scale)",
                  yaxis_type="log", template="plotly_white", width=850, height=450)
fig.show()

The loss falls in a straight line on the log scale, which means it shrinks by roughly the same fraction every epoch. For this data that can be checked by hand: the gradient is $15(w-2)$, because $\frac{2}{n}\sum x_i^2 = \frac{2 \cdot 30}{4} = 15$. Each update therefore multiplies the error $(w-2)$ by $(1 - 15\eta) = 0.85$ when $\eta = 0.01$.

## 2. What happens without `w.grad.zero_()`

`loss.backward()` does not overwrite `w.grad`; it **adds** the new gradient to whatever is already stored. If step 5 is removed, the value used in epoch $t$ is the sum of every gradient computed so far, not the current one. The run below repeats Challenge A with the same starting weight and learning rate, changing only that line.

In [4]:
w_no_reset, loss_no_reset = train_manual(lr=0.01, epochs=200, reset_grad=False)
print(f"With zero_():    final weight = {w_manual.item():.6f}, final loss = {loss_manual[-1]:.2e}")
print(f"Without zero_(): final weight = {w_no_reset.item():.6f}, final loss = {loss_no_reset[-1]:.2e}")

epochs_axis = list(range(1, 201))
fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs_axis, y=loss_manual, mode="lines", name="With w.grad.zero_()"))
fig.add_trace(go.Scatter(x=epochs_axis, y=loss_no_reset, mode="lines", name="Without w.grad.zero_()"))
fig.update_layout(title="Effect of not resetting gradients (same start, lr = 0.01)",
                  xaxis_title="Epoch", yaxis_title="MSE loss (log scale)",
                  yaxis_type="log", template="plotly_white", width=850, height=450)
fig.show()

With zero_():    final weight = 2.000000, final loss = 8.99e-13
Without zero_(): final weight = 2.432230, final loss = 8.26e-01


In [5]:
# First 12 epochs without the reset: the stored gradient keeps growing while the weight overshoots 2
w = w_init.clone().requires_grad_(True)
print("epoch | weight before | stored w.grad")
for epoch in range(12):
    loss = ((x @ w - y) ** 2).mean()
    loss.backward()
    print(f"{epoch + 1:5d} | {w.item():13.4f} | {w.grad.item():13.4f}")
    with torch.no_grad():
        w -= 0.01 * w.grad

epoch | weight before | stored w.grad
    1 |        1.5410 |       -6.8851
    2 |        1.6098 |      -12.7374
    3 |        1.7372 |      -16.6791
    4 |        1.9040 |      -18.1189
    5 |        2.0852 |      -16.8409
    6 |        2.2536 |      -13.0368
    7 |        2.3840 |       -7.2771
    8 |        2.4567 |       -0.4259
    9 |        2.4610 |        6.4892
   10 |        2.3961 |       12.4309
   11 |        2.2718 |       16.5080
   12 |        2.1067 |       18.1089


**Why the gradients pile up.** Without the reset, `w.grad` keeps the memory of all earlier steps. While the weight is below 2 every new gradient is negative, so the stored value keeps growing in size and the steps get longer even as the weight approaches 2. The weight passes 2, and only after several epochs of positive gradients does the stored sum change sign and pull it back. The table shows this: the weight overshoots to about 2.46 while the stored gradient is still negative.

The run never settles. Writing $e_t = w_t - 2$, the stored gradient is $15\sum_{k \le t} e_k$, which gives $e_{t+1} - 2e_t + e_{t-1} = -15\eta\, e_t$. That is the equation of an undamped oscillation: with $\eta = 0.01$ the error swings around 0 with a period of about 16 epochs, and its amplitude does not shrink. In the chart the loss keeps bouncing between about 1 and much smaller values for all 200 epochs, while the correct version reaches the precision limit of float32 (about $10^{-12}$) by epoch 90. The final values printed above confirm it. This is why PyTorch training loops call `optimizer.zero_grad()` (or `w.grad.zero_()`) on every iteration.

## 3. Challenge A with `torch.optim.SGD` and `nn.MSELoss`

The same model written the way PyTorch code is usually written: the weight is an `nn.Parameter`, `nn.MSELoss` computes the loss and `torch.optim.SGD` performs the update and the reset. It starts from the same weight with the same learning rate and number of epochs, so it should learn exactly the same value.

In [6]:
w_param = nn.Parameter(w_init.clone())
criterion = nn.MSELoss()
optimizer = torch.optim.SGD([w_param], lr=0.01)
loss_sgd = []

for epoch in range(200):
    loss = criterion(x @ w_param, y)   # forward pass + MSE in one line
    optimizer.zero_grad()              # replaces w.grad.zero_()
    loss.backward()                    # same autograd call as before
    optimizer.step()                   # replaces the manual no_grad update
    loss_sgd.append(loss.item())

print(f"Manual loop weight:  {w_manual.item():.8f}")
print(f"optim.SGD weight:    {w_param.item():.8f}")
print(f"Absolute difference: {abs(w_manual.item() - w_param.item()):.2e}")
print("Same weight:", torch.allclose(w_manual, w_param.detach()))
print("Same loss in every epoch:", torch.allclose(torch.tensor(loss_manual), torch.tensor(loss_sgd)))

Manual loop weight:  1.99999964
optim.SGD weight:    1.99999964
Absolute difference: 0.00e+00
Same weight: True
Same loss in every epoch: True


Both versions give the same weight and the same loss at every epoch. `optim.SGD` with no momentum applies exactly $w \leftarrow w - \eta \nabla_w L$, and `nn.MSELoss` uses the mean by default, just like `.mean()` in the manual loop. The optimizer only hides the update and reset steps; the mathematics does not change.

## 4. Challenge A with three learning rates

The same training run with $\eta = 0.001$, $0.01$ and $0.05$, all starting from the same weight for 200 epochs.

In [7]:
learning_rates = [0.001, 0.01, 0.05]
lr_results = {lr: train_manual(lr=lr, epochs=200) for lr in learning_rates}

print("learning rate | learned weight | final loss | first epoch with loss < 1e-6")
for lr, (w_lr, losses) in lr_results.items():
    first = next((i + 1 for i, v in enumerate(losses) if v < 1e-6), "not reached")
    print(f"{lr:13} | {w_lr.item():14.6f} | {losses[-1]:10.2e} | {first}")

fig = go.Figure()
for lr, (w_lr, losses) in lr_results.items():
    fig.add_trace(go.Scatter(x=epochs_axis, y=losses, mode="lines", name=f"lr = {lr}"))
fig.update_layout(title="Challenge A: loss for three learning rates",
                  xaxis_title="Epoch", yaxis_title="MSE loss (log scale)",
                  yaxis_type="log", template="plotly_white", width=850, height=450)
fig.show()

learning rate | learned weight | final loss | first epoch with loss < 1e-6
        0.001 |       1.977661 |   3.86e-03 | not reached
         0.01 |       2.000000 |   8.99e-13 | 45
         0.05 |       2.000000 |   0.00e+00 | 7


With this data each step multiplies the error by $(1 - 15\eta)$: about 0.985 for $\eta = 0.001$, 0.85 for $\eta = 0.01$ and 0.25 for $\eta = 0.05$. That is why the smallest learning rate is still away from the minimum after 200 epochs, while the largest reaches it in a few epochs. The flat part of the $\eta = 0.01$ curve near $10^{-12}$ is the precision limit of float32, not a problem with the optimization; with $\eta = 0.05$ the loss becomes exactly 0, so its line leaves the log-scale chart. The same formula shows the limit: for $\eta > 2/15 \approx 0.133$ the factor is below −1, so each step overshoots by more than it corrects and the loss grows instead of falling.

## 5. Reflection questions

### Q6. What happens when gradients are very small?

The update is the learning rate times the gradient, so a very small gradient means a very small step and training slows down or appears to stop. This has two very different causes. Near a minimum, a small gradient is the expected sign of convergence: in Challenge A the gradient is $15(w-2)$, so it shrinks towards zero exactly as $w$ approaches 2. Far from a minimum, a small gradient is a problem: the loss is still high but the model barely moves. This happens on flat regions of the loss surface and, in deep networks, through vanishing gradients, where the chain rule multiplies many factors smaller than one (for example saturated sigmoid or tanh units). A similar slow-progress effect appears when the learning rate is too small: with $\eta = 0.001$ the gradient was still large, but each step was so short that the weight was still clearly away from 2 after 200 epochs. Common remedies are a larger or adaptive learning rate, activations such as ReLU, careful initialization and normalization.

### Q9. Why do we reset gradients in PyTorch before the next iteration?

Because `backward()` accumulates: it adds the new gradient to the value already stored in `.grad` instead of replacing it. PyTorch does this on purpose, so gradients from several mini-batches can be summed before one update (useful when a large batch does not fit in memory). In a normal loop, however, each update should use only the gradient of the current loss. Section 2 shows what happens otherwise: the stored gradient keeps the memory of earlier steps, the weight overshoots and then keeps oscillating around 2 without converging (final loss about 0.83 instead of about $10^{-12}$). `w.grad.zero_()` or `optimizer.zero_grad()` restores the correct update rule.

### Q10. In PyTorch, how do `loss.backward()` and `optimizer.step()` divide the work?

`loss.backward()` only computes gradients: it runs reverse-mode automatic differentiation through the graph built during the forward pass and writes $\partial L / \partial \theta$ into the `.grad` field of every parameter. It does not change any parameter. `optimizer.step()` only updates parameters: it reads those `.grad` values and applies its update rule, which for plain SGD is $\theta \leftarrow \theta - \eta \, \theta.\text{grad}$. Section 3 confirms the split, because replacing the manual `no_grad` update with `optimizer.step()` gave the same weight to eight decimals. Keeping the two steps separate means the gradient computation never has to change when switching optimizers (SGD, Adam, momentum), and it makes room for steps between them, such as gradient clipping or accumulation.

## 6. Insights from the PyTorch runs

In [8]:
from IPython.display import Markdown, display

lr_small = lr_results[0.001]
lr_large = lr_results[0.05]
first_large = next(i + 1 for i, v in enumerate(lr_large[1]) if v < 1e-6)
display(Markdown(f'''
1. **The learning rate can be predicted for a simple model.** Because the gradient is $15(w-2)$, the error shrinks by a fixed factor $(1-15\\eta)$ per step. The runs match this: with $\\eta=0.05$ the loss dropped below $10^{{-6}}$ at epoch {first_large}, while with $\\eta=0.001$ the weight was still {lr_small[0].item():.4f} after 200 epochs. In larger networks this curvature is unknown, which is why the learning rate has to be tuned by experiment.
2. **A training bug does not always crash the code.** Removing `w.grad.zero_()` raised no error and the weight stayed close to the right region ({w_no_reset.item():.4f} at the end), but the loss was still {loss_no_reset[-1]:.2f} after 200 epochs, against {loss_manual[-1]:.1e} for the correct loop. Only the loss curve shows that the weight is swinging around 2 instead of converging. Plotting the loss every epoch is a cheap way to detect this kind of silent error.
3. **High-level PyTorch code is the same algorithm.** `nn.MSELoss` plus `optim.SGD` reproduced the manual loop exactly (difference {abs(w_manual.item() - w_param.item()):.1e}). Writing the manual version first makes it clear what `zero_grad`, `backward` and `step` actually do, which helps when debugging bigger models.
'''))


1. **The learning rate can be predicted for a simple model.** Because the gradient is $15(w-2)$, the error shrinks by a fixed factor $(1-15\eta)$ per step. The runs match this: with $\eta=0.05$ the loss dropped below $10^{-6}$ at epoch 7, while with $\eta=0.001$ the weight was still 1.9777 after 200 epochs. In larger networks this curvature is unknown, which is why the learning rate has to be tuned by experiment.
2. **A training bug does not always crash the code.** Removing `w.grad.zero_()` raised no error and the weight stayed close to the right region (2.4322 at the end), but the loss was still 0.83 after 200 epochs, against 9.0e-13 for the correct loop. Only the loss curve shows that the weight is swinging around 2 instead of converging. Plotting the loss every epoch is a cheap way to detect this kind of silent error.
3. **High-level PyTorch code is the same algorithm.** `nn.MSELoss` plus `optim.SGD` reproduced the manual loop exactly (difference 0.0e+00). Writing the manual version first makes it clear what `zero_grad`, `backward` and `step` actually do, which helps when debugging bigger models.
